In [ ]:
'''
First of all, get all the column names, and break them into categorical or numerical or timeseries. For categorical data, count all values. For numerical data, get the mean, median, mode, and the range, as well as quintiles. For time series, get the range, the mean median mode, and possibly graph a number? 
Quick graphs could be useful – graph order date vs ship date? Parse out by region? Could be a useful inclusion. In general, the first step is to understand what the hell we’re looking at. Simply looking at the dataset, we can tell this is sales data for a few year period. About 10,000 rows, okay. And then yes, look for missing data values in each column, and determine if it’s possible to impute them. Can also do checks – does every person have the same data (customer id always = same customer name, ship modes always equal across orders, always only to one location per order, order id always corresponds to same order-date / ship date.  Customer id always to same location? 
Also, I note I would collapse into several tables here – order Id is already sufficient key, don’t need multiple entries for each one, can have a separate table for that. Similarly, product id and product name dont need to both show up, nor do all of country/city/region/state/postal code. Customer ID also could be collapsed into its own table, especially if customers have a fixed address.
'''
import os
import sys
import json
import argparse
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#step 1: import data
df = pd.read_csv("GBP_DataSource1.csv", index_col=0, encoding='latin-1')
df.head(10)

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country/Region,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
Row ID,,,,,,,,,,,,,,,,,,,,
1.0,CA-2020-152156,08-11-2020,11-11-2020,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
2.0,CA-2020-152156,08-11-2020,11-11-2020,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
3.0,CA-2020-138688,12-06-2020,16-06-2020,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
4.0,US-2019-108966,11-10-2019,18-10-2019,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
5.0,US-2019-108966,11-10-2019,18-10-2019,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164
6.0,CA-2018-115812,09-06-2018,14-06-2018,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,FUR-FU-10001487,Furniture,Furnishings,Eldon Expressions Wood and Plastic Desk Access...,48.8600,7,0.00,14.1694
7.0,CA-2018-115812,09-06-2018,14-06-2018,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-AR-10002833,Office Supplies,Art,Newell 322,7.2800,4,0.00,1.9656
8.0,CA-2018-115812,09-06-2018,14-06-2018,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,TEC-PH-10002275,Technology,Phones,Mitel 5320 IP Phone VoIP phone,907.1520,6,0.20,90.7152
9.0,CA-2018-115812,09-06-2018,14-06-2018,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032.0,West,OFF-BI-10003910,Office Supplies,Binders,DXL Angle-View Binders with Locking Rings by S...,18.5040,3,0.20,5.7825


In [ ]:
#understand columns
print(df.columns)

#many helper functions to do this more easily
def infer_column_kinds(
    df: pd.DataFrame,
    *,
    cat_unique_ratio: float = 0.10,     # <= 10% unique (relative to non-null rows) tends to be categorical
    cat_unique_max: int = 50,           # or small absolute unique count
    parse_datetimes: bool = True,
    datetime_sample: int = 2000,        # sample size for datetime parsing attempt
) -> dict[str, list[str]]:
    """
    Return dict with keys: numeric, categorical, datetime, other.
    Uses dtype + lightweight heuristics (unique ratio / count) for categoricals.
    Optionally attempts to parse object columns as datetimes.
    """
    out = {"numeric": [], "categorical": [], "datetime": [], "other": []}

    # Start with dtype-based classification
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    dt_cols = df.select_dtypes(include=["datetime64[ns]", "datetimetz"]).columns.tolist()

    # Object/string/bool/category candidates
    remaining = [c for c in df.columns if c not in set(numeric_cols) | set(dt_cols)]

    # Try parse datetime for object-like columns (strict DD-MM-YYYY only)
    parsed_as_dt = set()
    if parse_datetimes:
        for c in remaining:
            s = df[c]
            if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                sample = s.dropna()
                if len(sample) == 0:
                    continue
                if len(sample) > datetime_sample:
                    sample = sample.sample(datetime_sample, random_state=0)

                # STRICT: must match exactly like 08-11-2020 (DD-MM-YYYY)
                parsed = pd.to_datetime(sample, errors="coerce", format="%d-%m-%Y")
                if parsed.notna().all():
                    parsed_as_dt.add(c)

    # finalize datetime
    dt_cols = dt_cols + sorted(parsed_as_dt)
    out["datetime"].extend(dt_cols)

    # numeric
    out["numeric"].extend(numeric_cols)

    # classify remaining as categorical vs other
    for c in df.columns:
        if c in out["numeric"] or c in out["datetime"]:
            continue

        s = df[c]
        # treat bool as categorical (often more useful)
        if pd.api.types.is_bool_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
            out["categorical"].append(c)
            continue

        # object/string: heuristic on unique counts
        non_null = s.dropna()
        n = len(non_null)
        if n == 0:
            out["other"].append(c)
            continue
        nunique = non_null.nunique(dropna=True)
        ratio = nunique / n if n else 0.0

        if (nunique <= cat_unique_max) or (ratio <= cat_unique_ratio):
            out["categorical"].append(c)
        else:
            # could be free text / ids / etc.
            out["other"].append(c)

    # keep stable order as in df
    for k in out:
        out[k] = [c for c in df.columns if c in set(out[k])]
    return out


def _safe_mode(series: pd.Series):
    s = series.dropna()
    if len(s) == 0:
        return np.nan
    m = s.mode(dropna=True)
    if len(m) == 0:
        return np.nan
    # if multiple modes, return list for honesty
    return m.iloc[0] if len(m) == 1 else m.tolist()


def summarize_numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    if not cols:
        return pd.DataFrame()

    rows = []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        s_non = s.dropna()
        if len(s_non) == 0:
            rows.append({"column": c})
            continue

        q = s_non.quantile([0.2, 0.4, 0.5, 0.6, 0.8])
        rows.append({
            "column": c,
            "count": int(s_non.size),
            "missing": int(df[c].isna().sum()),
            "mean": float(s_non.mean()),
            "median": float(s_non.median()),
            "mode": _safe_mode(s_non),
            "min": float(s_non.min()),
            "max": float(s_non.max()),
            "range": float(s_non.max() - s_non.min()),
            "q20": float(q.loc[0.2]),
            "q40": float(q.loc[0.4]),
            "q50": float(q.loc[0.5]),
            "q60": float(q.loc[0.6]),
            "q80": float(q.loc[0.8]),
            "std": float(s_non.std(ddof=1)) if s_non.size > 1 else 0.0,
        })
    return pd.DataFrame(rows).set_index("column")


def summarize_categorical(df: pd.DataFrame, cols: list[str], *, top_n: int | None = None) -> tuple[pd.DataFrame, dict[str, pd.Series]]:
    """
    Returns:
      - a per-column summary dataframe
      - a dict mapping column -> value_counts series (optionally truncated to top_n)
    """
    if not cols:
        return pd.DataFrame(), {}

    rows = []
    counts = {}
    for c in cols:
        s = df[c]
        non = s.dropna()
        vc = non.value_counts(dropna=False)
        counts[c] = vc.head(top_n) if top_n else vc

        rows.append({
            "column": c,
            "count": int(non.size),
            "missing": int(s.isna().sum()),
            "distinct": int(non.nunique(dropna=True)),
            "mode": _safe_mode(non),
            "mode_count": int(vc.iloc[0]) if len(vc) else 0,
        })

    return pd.DataFrame(rows).set_index("column"), counts


def summarize_datetime(df: pd.DataFrame, cols: list[str]) -> tuple[pd.DataFrame, dict[str, pd.Series]]:
    """
    Returns:
      - per-column datetime summary
      - dict mapping column -> counts per year (Series indexed by year)
    """
    if not cols:
        return pd.DataFrame(), {}

    rows = []
    per_year = {}
    for c in cols:
        s = df[c]
        # if it's object but inferred dt earlier, parse it
        if not (pd.api.types.is_datetime64_any_dtype(s) or isinstance(s.dtype, pd.DatetimeTZDtype)):
            s = pd.to_datetime(s, errors="coerce", format="%d-%m-%Y")

        s_non = s.dropna()
        if len(s_non) == 0:
            rows.append({"column": c})
            per_year[c] = pd.Series(dtype=int)
            continue

        # year breakdown
        years = s_non.dt.year
        year_counts = years.value_counts().sort_index()
        per_year[c] = year_counts

        # "mode" for datetimes is often not super informative unless timestamps repeat; still included
        rows.append({
            "column": c,
            "count": int(s_non.size),
            "missing": int(df[c].isna().sum()),
            "min": s_non.min(),
            "max": s_non.max(),
            "range_days": float((s_non.max() - s_non.min()) / np.timedelta64(1, "D")),
            "median": s_non.median(),
            "mode": _safe_mode(s_non),
            "distinct_years": int(year_counts.index.nunique()),
            "years": sorted(year_counts.index.tolist()),
        })

    return pd.DataFrame(rows).set_index("column"), per_year


def profile_dataframe(df: pd.DataFrame, *, top_n_categories: int | None = None) -> dict:
    kinds = infer_column_kinds(df)
    num_summary = summarize_numeric(df, kinds["numeric"])
    cat_summary, cat_counts = summarize_categorical(df, kinds["categorical"], top_n=top_n_categories)
    dt_summary, dt_per_year = summarize_datetime(df, kinds["datetime"])

    return {
        "kinds": kinds,
        "numeric_summary": num_summary,
        "categorical_summary": cat_summary,
        "categorical_value_counts": cat_counts,
        "datetime_summary": dt_summary,
        "datetime_points_per_year": dt_per_year,
        "other_columns": kinds["other"],
    }


Index(['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID',
       'Customer Name', 'Segment', 'Country/Region', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')


## Splitting rows by type of data

In [ ]:
report = profile_dataframe(df, top_n_categories=None)
report["kinds"]  # see which columns got classified as what

{'numeric': ['Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit'],
 'categorical': ['Ship Mode',
  'Customer ID',
  'Customer Name',
  'Segment',
  'Country/Region',
  'City',
  'State',
  'Region',
  'Category',
  'Sub-Category'],
 'datetime': ['Order Date', 'Ship Date'],
 'other': ['Order ID', 'Product ID', 'Product Name']}

## Summarizing Numerical Data

In [ ]:
report["numeric_summary"]           # mean/median/mode/range/quintiles/etc

,count,missing,mean,median,mode,min,max,range,q20,q40,q50,q60,q80,std
column,,,,,,,,,,,,,,
Postal Code,10009,11,55212.239185,56301.0000,10035.00,1040.000,99301.000,98261.000,19140.0000,43229.0000,56301.0000,75217.0000,90049.00000,32035.380091
Sales,10020,0,229.615136,54.3760,12.96,0.444,22638.480,22638.036,13.7732,34.2400,54.3760,89.7024,280.90160,622.572942
Quantity,10020,0,3.789421,3.0000,3.00,1.000,14.000,13.000,2.0000,3.0000,3.0000,4.0000,5.00000,2.224851
Discount,10020,0,0.156217,0.2000,0.00,0.000,0.800,0.800,0.0000,0.0000,0.2000,0.2000,0.20000,0.206460
Profit,10020,0,28.642121,8.6436,0.00,-6599.978,8399.976,14999.954,0.4068,5.4432,8.6436,13.4838,41.08496,233.971055


## Summarizing Categorical Data


In [ ]:
report["categorical_summary"]       # mode + distinct + missing


,count,missing,distinct,mode,mode_count
column,,,,,
Ship Mode,10019,1,4,Standard Class,5984
Customer ID,10020,0,793,WB-21850,37
Customer Name,10019,1,795,William Brown,37
Segment,10018,2,3,Consumer,5209
Country/Region,10018,2,2,United States,10016
City,10020,0,531,New York City,915
State,10019,1,49,California,2001
Region,10020,0,4,West,3207
Category,10020,0,3,Office Supplies,6043


## Datetime Summary

In [ ]:
report["datetime_summary"]          # min/max/range/median/mode/distinct years

,count,missing,min,max,range_days,median,mode,distinct_years,years
column,,,,,,,,,
Order Date,10018,2,2018-01-03,2021-12-30,1457.0,2020-06-26,2020-09-05,4,"[2018, 2019, 2020, 2021]"
Ship Date,10019,1,2018-01-07,2022-01-05,1459.0,2020-06-29,2019-12-16,5,"[2018, 2019, 2020, 2021, 2022]"


In [ ]:
#also, orders per year
report["datetime_points_per_year"]  # dict: col -> counts per year


{'Order Date': Order Date
 2018    1998
 2019    2105
 2020    2598
 2021    3317
 Name: count, dtype: int64,
 'Ship Date': Ship Date
 2018    1945
 2019    2136
 2020    2588
 2021    3308
 2022      42
 Name: count, dtype: int64}

In [ ]:
'''
next we want to clean up some data. 
based on the missing values, it seems like we can easily just fill in based on nearby data
but we need to make sure that the relationships we want to fill on are actually consistent
e,g, is order date always consistent with order number? 
first we'll assess the damage, then draw conclusions

'''
rows_with_missing = df[df.isna().any(axis=1)]
print(len(rows_with_missing))
print(rows_with_missing)
print(rows_with_missing.index)

23
              Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
Row ID                                                                       
2235.0  CA-2021-104066  05-12-2021  10-12-2021  Standard Class    QJ-19255   
5275.0  CA-2019-162887  07-11-2019  09-11-2019    Second Class    SV-20785   
8799.0  US-2020-150140  06-04-2020  10-04-2020  Standard Class    VM-21685   
8831.0  CA-2020-167983  21-08-2020         NaN    Second Class    RP-19270   
8893.0  CA-2019-111038         NaN  06-12-2019  Standard Class    LC-16960   
8938.0             NaN  17-12-2019  19-12-2019    Second Class    PG-18895   
9022.0  CA-2019-129525         NaN  20-11-2019  Standard Class    VP-21760   
9147.0  US-2020-165505  23-01-2020  27-01-2020  Standard Class    CB-12535   
9148.0  US-2020-165505  23-01-2020  27-01-2020  Standard Class    CB-12535   
9149.0  US-2020-165505  23-01-2020  27-01-2020  Standard Class    CB-12535   
9181.0  CA-2021-156776  07-08-2021  11-08-2021  Standard Clas

In [ ]:
'''
There are 23 rows with missing data. Let's go column by column and see how we can impute.
Row ID: assumed to simply increment. Just look at neighbours. 
Order ID: if customer, order, and ship date match, can impute
Order Date/Ship Date: reference order ID (but check if this relationship holds)
Ship Mode can also be included with the above
Customer ID/Customer Name: We assume this is linked with Customer Name always, otherwise no way to impute
Segment: we assume this is always liniked to customer ID (but we check)
Country/Region: Impute Via State? There are two vlaues for Canada here, otheriwse it's all US
Okay look I just manually checked and the two cities have state matching zip codes so. imputed as such. 
City: do we...have to check if every city has the right state? 
Postal Code: Is every postal code limited to one state/city? ffs
Region: same check here for city, state, postal code.
We want to check: does postal code always match one city, one state, one region
Does state always match region
Does city always match postal code?
'''


0


In [ ]:
#checking order - order date/shipdate correspondence
tmp = df.dropna(subset=["Order ID", "Order Date", "Ship Date", "Ship Mode"]).copy()
bad = tmp.groupby("Order ID")[["Order Date", "Ship Date", "Ship Mode"]].nunique().gt(1).any(axis=1)
bad_order_ids = bad[bad].index

# inspect offending rows (if any)
order_mismatches = tmp[tmp["Order ID"].isin(bad_order_ids)].sort_values(["Order ID", "Order Date", "Ship Date", "Ship Mode"])
print(len(order_mismatches))
#it's 0, so we can conclude these always go together

0


In [ ]:
#checking customer ID - segment correspondence
cols = ["Customer ID", "Customer Name", "Segment"]
tmp2 = df.loc[df[cols].notna().all(axis=1), cols].copy()
bad2 = tmp2.groupby("Customer ID")[["Customer Name", "Segment"]].nunique().gt(1).any(axis=1)
bad_order_ids2 = bad2[bad2].index

# inspect offending rows (if any)
order_mismatches2 = tmp2[tmp2["Customer ID"].isin(bad_order_ids2)].sort_values(["Customer ID", "Customer Name", "Segment"])
print(order_mismatches2)
#alright so we find that Customer name doesnt always track customer Id, some are missing lastname, easily imputed


       Customer ID    Customer Name      Segment
Row ID                                          
9855.0    GA-14515          George      Consumer
2753.0    GA-14515  George Ashbrook     Consumer
2754.0    GA-14515  George Ashbrook     Consumer
2755.0    GA-14515  George Ashbrook     Consumer
2756.0    GA-14515  George Ashbrook     Consumer
2757.0    GA-14515  George Ashbrook     Consumer
3525.0    GA-14515  George Ashbrook     Consumer
4050.0    GA-14515  George Ashbrook     Consumer
4051.0    GA-14515  George Ashbrook     Consumer
7088.0    GA-14515  George Ashbrook     Consumer
7729.0    GA-14515  George Ashbrook     Consumer
9833.0    GA-14515  George Ashbrook     Consumer
9856.0    GA-14515  George Ashbrook     Consumer
9857.0    GA-14515  George Ashbrook     Consumer
9860.0    HR-14770          Hallie   Home Office
1025.0    HR-14770   Hallie Redmond  Home Office
5035.0    HR-14770   Hallie Redmond  Home Office
5714.0    HR-14770   Hallie Redmond  Home Office
9443.0    HR-14770  

In [ ]:
#we do a separate check for just segment and customer id relation and find its empty. nice!
cols = ["Customer ID", "Segment"]
tmp2 = df.loc[df[cols].notna().all(axis=1), cols].copy()
bad2 = tmp2.groupby("Customer ID")[["Segment"]].nunique().gt(1).any(axis=1)
bad_order_ids2 = bad2[bad2].index

# inspect offending rows (if any)
order_mismatches2 = tmp2[tmp2["Customer ID"].isin(bad_order_ids2)].sort_values(["Customer ID", "Segment"])
print(order_mismatches2)
#sad, empty df

Empty DataFrame
Columns: [Customer ID, Segment]
Index: []


In [ ]:
'''Reminder:
Country/Region: Impute Via State? There are two vlaues for Canada here, otheriwse it's all US
Okay look I just manually checked and the two cities have state matching zip codes so. imputed as such. 
City: do we...have to check if every city has the right state? 
Postal Code: Is every postal code limited to one state/city? ffs
Region: same check here for city, state, postal code.
We want to check: does postal code always match one city, one state, one region
Does state always match region
Does city always match postal code?
'''
#checking that everything matches postal codes...god help us
cols = ["Country/Region", "City", "State", "Postal Code", "Region"]
tmp2 = df.loc[df[cols].notna().all(axis=1), cols].copy()
tmp2 = tmp2[~tmp2["Country/Region"].isin(["Canada"])].copy()
tmp2 = tmp2[~tmp2["City"].isin(["Encinitas"])].copy()
bad2 = tmp2.groupby("Postal Code")[["Country/Region", "City", "State", "Region"]].nunique().gt(1).any(axis=1)
bad_order_ids2 = bad2[bad2].index

# inspect offending rows (if any)
order_mismatches2 = tmp2[tmp2["Postal Code"].isin(bad_order_ids2)].sort_values(["Country/Region", "City", "State", "Postal Code", "Region"])
print(order_mismatches2)
#city encinitas is sharing postal code with san diego, otherwise we're gucchi. this isnt erroneous seemingly


Empty DataFrame
Columns: [Country/Region, City, State, Postal Code, Region]
Index: []


In [ ]:
''' 
Last remaining columns!
Product ID, Category, Sub-category, Product Name
these probably dont all match too but the fuck can we do. 
probably assume product idc always matches product name
then from there make sure product id always matches category and sub category. cool. 
'''
#checking that everything matches postal codes...god help us
cols = ["Product ID", "Product Name"]
tmp2 = df.loc[df[cols].notna().all(axis=1), cols].copy()
bad2 = tmp2.groupby("Product ID")[["Product Name"]].nunique().gt(1).any(axis=1)
bad_order_ids2 = bad2[bad2].index
# inspect offending rows (if any)
order_mismatches2 = tmp2[tmp2["Product ID"].isin(bad_order_ids2)].sort_values(["Product ID", "Product Name"])
pd.set_option("display.max_rows", 350)        # or None
print(order_mismatches2)

             Product ID                                       Product Name
Row ID                                                                    
2116.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
5919.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
6536.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
9396.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
9584.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
9650.0  FUR-BO-10002213              DMI Eclipse Executive Suite Bookcases
2472.0  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish
2809.0  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish
5080.0  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish
8713.0  FUR-BO-10002213   Sauder Forest Hills Library, Woodland Oak Finish
129.0   FUR-CH-10001146                           Global Task Chair, Black
6743.0  FUR-CH-10001146  

In [ ]:
''' 
Okay, there are a million fucking products that share a product id. 
I guess we cna write a more involved script that identifies if any both share a product ID 
AND have their own distinct product id, and correct those
and then for the rest, we just. like. add a 1 or 2 to their product id (or a 3 god forbid)
annoying but manageable. ive run out of time to do this however
good luck rest of team
'''